### Lexicon Builder


In [ ]:
import os
import sys
import re
import json
import pickle

import requests
import spacy 
import nltk

from collections import Counter
import pandas as pd
from tqdm import tqdm

sys.path.append('..')

from utils.json import load_json, save_json
from utils.dataset import extract_annotations

# TODO Implement manual labeling of languages
# - Alternatives to FastText
# - Manual Labeling
# - Not crucial but useful

In [ ]:
tqdm.pandas()

In [ ]:
# !python -m spacy download es_core_news_sm
# !python -m spacy download ca_core_news_sm

In [ ]:
# Define the paths
path_annotations = "../data/annotations"
path_lexicons = "../data/lexicons"
path_negation = "../data/lexicons/negation"
path_uncertainty = "../data/lexicons/uncertainty"
path_model = "../data/lexicon/models"

In [ ]:
"""
# -- FastText Language Detection (Not works as expected) --

# !pip install fasttext spacy
# !python -m spacy download es_core_news_sm
# !python -m spacy download ca_core_news_sm

# URL of the FastText model
url = "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin"
 
# Path to fastText model
file_model = os.path.join(path_model, "lid.176.bin")

# Download fastText model if needed
if not os.path.exists(file_model):
    print("Downloading fastText language detection model...")
    url = "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin"
    response = requests.get(url, stream=True)
    response.raise_for_status() # Raise an exception
    with open(file_model, "wb") as f:
        f.write(response.content)
    print("Model downloaded successfully!")
else:
    print("FastText model already exists at:", file_model)
"""

In [ ]:
os.makedirs(path_negation, exist_ok=True)
os.makedirs(path_uncertainty, exist_ok=True)

In [ ]:
print("Loading spaCy models for POS tagging...")
try:
    nlp_es = spacy.load("es_core_news_sm")
    nlp_ca = spacy.load("ca_core_news_sm")
    print("Both Spanish and Catalan models loaded successfully")
except FileNotFoundError as e:
    print("Warning: Could not load spaCy models. Installing now...")
    # !python -m spacy download es_core_news_sm
    # !python -m spacy download ca_core_news_sm

In [ ]:
# Load the training dataframe
with open(os.path.join(path_annotations, "df_train.pkl"), "rb") as f:
    df_train = pickle.load(f)

display(df_train.head())

print("Label distribution:")
print(df_train["label"].value_counts())

In [ ]:
def clean_text(text):
    if not text:
        return ""
    
    text = text.lower().strip()
    text = re.sub(r"^[\.,;:\s]+|[\.,;:\s]+$", "", text) # Remove punctuation at beginning and end
    
    return text

In [ ]:
neg_df = df_train[df_train["label"] == "NEG"].copy() # Filter only NEG labels
unc_df = df_train[df_train["label"] == "UNC"].copy() # Filter only UNC labels
print(f"Found {len(neg_df)} negation cues and {len(unc_df)} uncertainty cues")


# Obtain unique negation and uncertainty cues
unique_neg_cues = set(neg_df["text"].apply(clean_text).unique())
unique_unc_cues = set(unc_df["text"].apply(clean_text).unique())
print(f"Unique negation cues: {len(unique_neg_cues)}")
print(f"Unique uncertainty cues: {len(unique_unc_cues)}")

In [ ]:
# Clean and normalize terms
neg_df["clean_text"] = neg_df["text"].apply(clean_text)
unc_df["clean_text"] = unc_df["text"].apply(clean_text)

neg_df = neg_df[neg_df["clean_text"] != ""] # Remove empty terms
unc_df = unc_df[unc_df["clean_text"] != ""] # Remove empty terms
print(f"After cleaning: {len(neg_df)} negation cues and {len(unc_df)} uncertainty cues")

neg_counts = neg_df["clean_text"].value_counts().to_dict() # Count term frequencies
unc_counts = unc_df["clean_text"].value_counts().to_dict() # Count term frequencies


neg_lexicon = pd.DataFrame({ # NEG Lexicon DataFrames with unique terms
    "term": list(neg_counts.keys()),
    "freq": list(neg_counts.values())
})

unc_lexicon = pd.DataFrame({ # UNC Lexicon DataFrames with unique terms
    "term": list(unc_counts.keys()),
    "freq": list(unc_counts.values())
})


print("\nTOP negation cues:")
for term, count in sorted(neg_counts.items(), key=lambda x: x[1], reverse=True)[:20]:
    print(f"{term}: {count}")

print("\nTOP uncertainty cues:")
for term, count in sorted(unc_counts.items(), key=lambda x: x[1], reverse=True)[:20]:
    print(f"{term}: {count}")

In [ ]:
def detect_language(text, nlp_es, nlp_ca):
    """
    Language detection using spaCy + rules
    
    Parameters:
        text (str): The text to detect
        nlp_es: Spanish spaCy model
        nlp_ca: Catalan spaCy model
        
    Returns:
        str: "ca" for Catalan, "es" for Spanish
    """
    # if not text or len(text) < 3:
    #     return "es"
    
    text = text.lower().strip()
    
    # Use spaCy models to analyze text
    # Measure "surprisal" - how unexpected the text is in each language
    
    try:
        # Process with both models
        doc_es = nlp_es(text)
        doc_ca = nlp_ca(text)
        
        es_recognized = sum(1 for token in doc_es if not token.is_oov) # Count tokens recognized by es model
        ca_recognized = sum(1 for token in doc_ca if not token.is_oov) # Count tokens recognized by cat model
        
        # If more tokens recognized by Catalan model, return Catalan
        if ca_recognized > es_recognized:
            return "ca"
        # If equal or more recognized by Spanish, return Spanish
        else:
            return "es"
    except:
        return "es"  # Default to Spanish on error


In [ ]:
# TODO - Improve the language detection

# Using spaCy-based approach 
print("Detecting languages for negation terms...")
neg_lexicon["language"] = neg_lexicon["term"].apply(lambda x: detect_language(x, nlp_es, nlp_ca))

print("Detecting languages for uncertainty terms...")
unc_lexicon["language"] = unc_lexicon["term"].apply(lambda x: detect_language(x, nlp_es, nlp_ca))

In [ ]:
def determine_POS(term, language):
    term = term.strip().lower()
    
    if not term:
        return "NA" # Special cases
    
    prefixes = ["in", "im", "i", "des", "dis", "a"]
    if term in prefixes or term.endswith("-"):
        return "prefix" 
    
    if term.startswith("-"):
        return "suffix"
    
    nlp = nlp_ca if language == "ca" else nlp_es # Appropriate spaCy model
    doc = nlp(term) # Process with spaCy
    
    # Map spaCy"s universal POS tags to categories
    pos_map = {
        "ADV": "adverb",
        "VERB": "verb",
        "ADP": "preposition",
        "DET": "determiner",
        "ADJ": "adjective",
        "NOUN": "noun",
        "PRON": "pronoun",
        "CCONJ": "conjunction",
        "SCONJ": "conjunction"
    }
    
    # Multi-word expressions
    if len(doc) > 1:
        roots = [token for token in doc if token.dep_ == "ROOT"]
        if roots:
            pos = roots[0].pos_
            return pos_map.get(pos, "phrase")
        else:
            return "phrase"
    
    # Single-word expressions
    if len(doc) == 1:
        pos = doc[0].pos_
        return pos_map.get(pos, "other")
    
    return "NA" # Default


# Apply POS tagging
print("Determining POS for negation terms...")
neg_lexicon["POS"] = neg_lexicon.apply(
    lambda row: determine_POS(row["term"], row["language"]), axis=1
)

print("Determining POS for uncertainty terms...")
unc_lexicon["POS"] = unc_lexicon.apply(
    lambda row: determine_POS(row["term"], row["language"]), axis=1
)

In [ ]:
# Sort lexicons by frequency
neg_lexicon = neg_lexicon.sort_values("freq", ascending=False).reset_index(drop=True)
unc_lexicon = unc_lexicon.sort_values("freq", ascending=False).reset_index(drop=True)

print("Negation lexicon preview:")
display(neg_lexicon.head())

print("Uncertainty lexicon preview:")
display(unc_lexicon.head())

In [ ]:
# Save complete lexicons to directory
neg_lexicon.to_csv(os.path.join(path_negation, "negation_ALL.csv"), index=False)
unc_lexicon.to_csv(os.path.join(path_uncertainty, "uncertainty_ALL.csv"), index=False)

# Language and save
neg_lexicon_es = neg_lexicon[neg_lexicon["language"] == "es"]
neg_lexicon_ca = neg_lexicon[neg_lexicon["language"] == "ca"]
unc_lexicon_es = unc_lexicon[unc_lexicon["language"] == "es"]
unc_lexicon_ca = unc_lexicon[unc_lexicon["language"] == "ca"]

In [ ]:
# Save to negation and uncertainty directories
neg_lexicon_es.to_csv(os.path.join(path_negation, "negation_es.csv"), index=False)
neg_lexicon_ca.to_csv(os.path.join(path_negation, "negation_ca.csv"), index=False)
unc_lexicon_es.to_csv(os.path.join(path_uncertainty, "uncertainty_es.csv"), index=False)
unc_lexicon_ca.to_csv(os.path.join(path_uncertainty, "uncertainty_ca.csv"), index=False)

print(f"Saved {len(neg_lexicon_es)} Spanish and {len(neg_lexicon_ca)} Catalan negation terms")
print(f"Saved {len(unc_lexicon_es)} Spanish and {len(unc_lexicon_ca)} Catalan uncertainty terms")

# Output language and POS distributions
print("\nNegation language distribution:")
print(neg_lexicon["language"].value_counts())

print("\nNegation POS distribution:")
print(neg_lexicon["POS"].value_counts())

print("\nUncertainty language distribution:")
print(unc_lexicon["language"].value_counts())

print("\nUncertainty POS distribution:")
print(unc_lexicon["POS"].value_counts())

print("\nLexicon building complete!")